In [4]:
import os
notebook_dir = '/workspaces/llm-zoomcamp-rag/llm-zoomcamp-onnx'
os.chdir(notebook_dir)

In [5]:
import sys
sys.path.insert(0, '/workspaces/llm-zoomcamp-rag/llm-zoomcamp-onnx')

In [ ]:

from embedder import Embedder
embed = Embedder('/workspaces/llm-zoomcamp-rag/llm-zoomcamp-onnx/models/Xenova/all-MiniLM-L6-v2')

q1 = "Can I still join the course after the start date?"
q2 = "How to install Docker on Windows?"
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

v1 = embed.encode(q1)
v2 = embed.encode(q2)
dv = embed.encode(d)

2026-06-26 00:37:37.466972668 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [7]:
v1.dot(dv)

np.float64(0.3233238425677314)

In [8]:
v2.dot(dv)

np.float64(0.01973045838992233)

In [9]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-06-26 00:37:39--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 888 [text/plain]
Saving to: ‘ingest.py.1’

ingest.py.1         100%[===================>]     888  --.-KB/s    in 0s      

2026-06-26 00:37:39 (48.4 MB/s) - ‘ingest.py.1’ saved [888/888]



In [10]:
from ingest import load_faq_data

documents = load_faq_data()

In [11]:
texts = [doc["question"] + " " + doc["answer"] for doc in documents]

In [12]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/27 [00:00<?, ?it/s]

In [13]:
query = "Can I still join the course after the start date?"
v_query = embed.encode(query)

scores = X.dot(v_query)
idx = np.argmax(scores)

documents[idx]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

## Q1: Embed the following query: How does approximate nearest neighbor search work?


In [14]:
query = "How does approximate nearest neighbor search work?"
v_query = embed.encode(query)
v_query[0]

np.float64(-0.02058203437252893)

### Q1 Answer: -0.02

## Q2: Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?

In [15]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [16]:
# 02-vector-search/lessons/07-sqlitesearch-vector.md
doc = next(doc for doc in documents if doc['filename'] == '02-vector-search/lessons/07-sqlitesearch-vector.md')

In [17]:
doc

{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done 

In [18]:
d1 = doc['content']

In [19]:
d1v = embed.encode(d1)

In [20]:
v_query.dot(d1v)

np.float64(0.36107027225589694)

### Q2 Answer: I got about 0.37

## Q3 Answer: We embed every chunk's content with encode_batch, stack the vectors into a matrix X, and score the Q1 query against all chunks. Which file does the highest-scoring chunk belong to (its filename)?

In [21]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [26]:
chunk_vectors = [embed.encode(chunk['content']) for chunk in chunks]
X = np.array(chunk_vectors)

In [27]:
scores = X.dot(v_query)

In [29]:
idx = np.argmax(scores)

In [30]:
chunks[idx]['filename']

'02-vector-search/lessons/07-sqlitesearch-vector.md'

### Q3 Answer: 02-vector-search/lessons/07-sqlitesearch-vector.md

## Q4: We've done vector search by hand, which is good for learning, but it's not what we do in practice. In practice we use libraries. Let's use VectorSearch from minsearch and run a search for the following query: What metric do we use to evaluate a search engine?

In [38]:
from minsearch import VectorSearch

In [39]:
vs_index = VectorSearch(keyword_fields=["filename"])

In [40]:
vs_index.fit(X, chunks)  # X is your matrix of chunk vectors from the previous step

In [41]:
query = "What metric do we use to evaluate a search engine?"
query_vector = embed.encode(query)

In [42]:
results = vs_index.search(query_vector, num_results=5)

In [44]:
results[0]['filename']

'04-evaluation/lessons/05-search-metrics.md'

### Q4 Answer: 04-evaluation/lessons/05-search-metrics.md

## Q5: Which file shows up in the vector results but not in the text results?

In [45]:
from minsearch import Index

# Text search index
text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

In [ ]:
query = "How do I store vectors in PostgreSQL?"

# Vector search 
query_vector = embed.encode(query)
vector_results = vs_index.search(query_vector, num_results=5)

In [48]:
# Text search
text_results = text_index.search(query, num_results=5)

In [49]:
# Compare filenames
vector_files = {r['filename'] for r in vector_results}
text_files = {r['filename'] for r in text_results}

In [50]:
print("In vector but not text:", vector_files - text_files)

In vector but not text: {'02-vector-search/lessons/08-pgvector.md'}


### Q5 answer: 02-vector-search/lessons/08-pgvector.md

## Q6: Hybrid search.Which file is ranked first after RRF?

In [51]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [55]:
query = "How do I give the model access to tools?"

In [56]:
query_vector = embed.encode(query)

vector_results = vs_index.search(query_vector, num_results=5)

In [57]:
text_results = text_index.search(query, num_results=5)

In [58]:
results = rrf([vector_results, text_results])

In [59]:
results[0]['filename']

'01-agentic-rag/lessons/13-function-calling.md'

### Q6 Answer: '01-agentic-rag/lessons/13-function-calling.md'